# Notebook 1 — IoT Sensor Data Generator

**Purpose:** Simulate raw factory sensor data and save it as a Bronze Delta table in Unity Catalog.

**What this notebook does:**
- Creates the catalog, schema and volume in Unity Catalog
- Generates 10,000 sensor readings across 5 machines
- Simulates 3 sensor types: temperature, pressure, vibration
- Intentionally injects nulls, duplicates and out-of-range values (to clean in Notebook 2)
- Saves raw data as a Delta table: `iot_pipeline.sensors.bronze_sensors`

**Output:** `iot_pipeline.sensors.bronze_sensors`


In [0]:
import random
from datetime import datetime, timedelta

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)

# Unity Catalog config
CATALOG_NAME = "iot_pipeline"
SCHEMA_NAME  = "sensors"

# Full table reference
BRONZE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_sensors"

# Seed for reproducibility
random.seed(42)

print("Config ready.")
print(f"Bronze table: {BRONZE_TABLE}")

In [0]:
# Creating catalog,schema and volume 
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.raw_data")

print(f"Catalog : {CATALOG_NAME} — ready")
print(f"Schema  : {CATALOG_NAME}.{SCHEMA_NAME} — ready")
print(f"Volume  : {CATALOG_NAME}.{SCHEMA_NAME}.raw_data — ready")

In [0]:
# 5 factory machines
MACHINES = ["MACHINE_01", "MACHINE_02", "MACHINE_03", "MACHINE_04", "MACHINE_05"]

# Normal operating ranges per sensor type
SENSOR_CONFIG = {
    "temperature": {"min": 60.0, "max": 120.0, "unit": "celsius"},
    "pressure":    {"min": 1.0,  "max": 10.0,  "unit": "bar"},
    "vibration":   {"min": 0.01, "max": 2.0,   "unit": "mm/s"},
}

# Simulation period: last 7 days
END_TIME   = datetime(2024, 1, 7, 23, 59, 59)
START_TIME = END_TIME - timedelta(days=7)

NUM_RECORDS = 10_000

print(f"Simulating  : {NUM_RECORDS:,} records")
print(f"Period      : {START_TIME} → {END_TIME}")
print(f"Machines    : {MACHINES}")
print(f"Sensor types: {list(SENSOR_CONFIG.keys())}")

In [0]:
def generate_sensor_record(record_id: int) -> dict:
    """
    Generate one sensor reading.
    Intentionally injects anomalies to make Notebook 2 meaningful:
      - 3%  chance of null sensor value   (missing data)
      - 2%  chance of duplicate event_id  (duplicates)
      - 5%  chance of out-of-range value  (faulty sensor spike)
    """
    machine_id  = random.choice(MACHINES)
    sensor_type = random.choice(list(SENSOR_CONFIG.keys()))
    cfg         = SENSOR_CONFIG[sensor_type]

    # Random timestamp within the simulation window
    offset_secs = random.randint(0, int((END_TIME - START_TIME).total_seconds()))
    ts          = START_TIME + timedelta(seconds=offset_secs)

    rand = random.random()

    if rand < 0.03:
        # Null value — missing reading
        value = None
    elif rand < 0.08:
        # Out-of-range spike — faulty sensor
        value = round(random.uniform(cfg["max"] * 1.5, cfg["max"] * 3.0), 4)
    else:
        # Normal reading with small Gaussian noise
        center = (cfg["min"] + cfg["max"]) / 2
        spread = (cfg["max"] - cfg["min"]) / 6
        value  = round(random.gauss(center, spread), 4)
        value  = round(max(cfg["min"], min(cfg["max"], value)), 4)

    # Occasionally reuse a previous event_id to simulate duplicates
    if rand < 0.02:
        event_id = f"EVT-{random.randint(1, record_id)}"
    else:
        event_id = f"EVT-{record_id}"

    return {
        "event_id":    event_id,
        "machine_id":  machine_id,
        "sensor_type": sensor_type,
        "value":       value,
        "unit":        cfg["unit"],
        "timestamp":   ts,
        "ingested_at": datetime.utcnow(),
    }

# Quick sanity check
sample = generate_sensor_record(1)
print("Sample record:")
for k, v in sample.items():
    print(f"  {k}: {v}")

In [0]:
print(f"Generating {NUM_RECORDS:,} records...")

records = [generate_sensor_record(i) for i in range(1, NUM_RECORDS + 1)]

# Count injected problems so we know what to expect in Notebook 2
nulls      = sum(1 for r in records if r["value"] is None)
duplicates = NUM_RECORDS - len({r["event_id"] for r in records})
out_range  = sum(
    1 for r in records
    if r["value"] is not None
    and r["value"] > SENSOR_CONFIG[r["sensor_type"]]["max"]
)

print(f"Done. Injected issues summary:")
print(f"  Null values   : {nulls:>5}  ({nulls/NUM_RECORDS*100:.1f}%)")
print(f"  Duplicate IDs : {duplicates:>5}  ({duplicates/NUM_RECORDS*100:.1f}%)")
print(f"  Out-of-range  : {out_range:>5}  ({out_range/NUM_RECORDS*100:.1f}%)")

In [0]:
schema = StructType([
    StructField("event_id",    StringType(),    nullable=False),
    StructField("machine_id",  StringType(),    nullable=False),
    StructField("sensor_type", StringType(),    nullable=False),
    StructField("value",       DoubleType(),    nullable=True),
    StructField("unit",        StringType(),    nullable=False),
    StructField("timestamp",   TimestampType(), nullable=False),
    StructField("ingested_at", TimestampType(), nullable=False),
])

df_bronze = spark.createDataFrame(records, schema=schema)

print(f"DataFrame created: {df_bronze.count():,} rows, {len(df_bronze.columns)} columns")
df_bronze.printSchema()

In [0]:
print("=== Sample rows ===")
df_bronze.show(10, truncate=False)

print("=== Row counts per machine ===")
df_bronze.groupBy("machine_id").count().orderBy("machine_id").show()

print("=== Row counts per sensor type ===")
df_bronze.groupBy("sensor_type").count().orderBy("sensor_type").show()

In [0]:
# Drop if exists — safe for re-runs
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("sensor_type")
    .saveAsTable(BRONZE_TABLE)    # Unity Catalog handles storage location
)

print(f"Bronze table saved: {BRONZE_TABLE}")

In [0]:
result = spark.sql(f"SELECT COUNT(*) AS total_rows FROM {BRONZE_TABLE}").collect()[0]
print(f"Table '{BRONZE_TABLE}' registered successfully.")
print(f"Total rows: {result['total_rows']:,}")

print("\n=== Sample from Unity Catalog table ===")
spark.sql(f"SELECT * FROM {BRONZE_TABLE} LIMIT 5").show(truncate=False)

print("\n=== Partition info ===")
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").select("format", "numFiles", "sizeInBytes").show()